# Notebook 02: Trigger Phrase Ablation

**Mục tiêu:** So sánh các trigger phrases khác nhau với Budget Forcing
- 'Wait' (default từ paper)
- 'Hmm, let me reconsider'
- 'Hold on'
- 'Let me double-check'
- 'Actually...'

**Research Gap từ paper:** Section 6.2 đề xuất nhưng chưa implement.

**Người thực hiện:** Người A  
**Timeline:** Ngày 8–9

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from models.model_loader import load_model_and_tokenizer
from budget_forcing import BudgetForcingDecoder, compute_all_metrics
from evaluation.run_eval import format_prompt, extract_answer, check_answer
from datasets import load_dataset
import random

# Load model (đã có từ notebook 01 — dùng lại)
MODEL_NAME = 'r1-distill-7B'  # ← same as notebook 01
model, tokenizer = load_model_and_tokenizer(MODEL_NAME, load_in_4bit=True)
decoder = BudgetForcingDecoder(model, tokenizer)

In [ ]:
# ── Trigger phrases to compare ─────────────────────────────────────────────
TRIGGERS = {
    'Wait':               'Wait',                      # Paper default
    'Hmm':                'Hmm, let me reconsider',    # More explicit reflection
    'Hold on':            'Hold on',                   # Short variant
    'Double-check':       'Let me double-check',       # Verification focus
    'Actually':           'Actually, wait...',          # Hedge variant
}

N_WAIT = 2       # Dùng n_wait=2 (consistent với paper ablation)
N_SAMPLES = 50   # Số câu MATH500

# Load data
random.seed(42)
math500 = load_dataset('HuggingFaceH4/MATH-500', split='test')
indices = random.sample(range(len(math500)), N_SAMPLES)
samples = [math500[i] for i in indices]

In [ ]:
# ── Run ablation ───────────────────────────────────────────────────────────
ablation_results = {}

for trigger_name, trigger_text in TRIGGERS.items():
    correct = 0
    thinking_tokens_list = []
    
    for i, sample in enumerate(tqdm(samples, desc=f'Trigger: {trigger_name}')):
        question = sample['problem']
        ground_truth = sample['answer']
        
        prompt = format_prompt(question)
        input_ids = tokenizer.encode(prompt, return_tensors='pt')
        
        output = decoder.generate(
            input_ids,
            max_new_tokens=2048,
            n_wait=N_WAIT,
            trigger=trigger_text,
        )
        
        predicted = extract_answer(output['answer_text'])
        is_correct = check_answer(predicted, ground_truth)
        correct += int(is_correct)
        thinking_tokens_list.append(output['thinking_tokens'])
    
    ablation_results[trigger_name] = {
        'trigger_text': trigger_text,
        'accuracy': correct / N_SAMPLES,
        'avg_thinking_tokens': np.mean(thinking_tokens_list),
        'std_thinking_tokens': np.std(thinking_tokens_list),
    }
    print(f'{trigger_name}: {correct/N_SAMPLES:.1%}')

print('\n=== Ablation Summary ===')
for k, v in sorted(ablation_results.items(), key=lambda x: -x[1]['accuracy']):
    print(f'{k:15s}: acc={v["accuracy"]:.1%}, avg_tokens={v["avg_thinking_tokens"]:.0f}')

In [ ]:
# ── Visualization ──────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

names = list(ablation_results.keys())
accs = [ablation_results[n]['accuracy'] * 100 for n in names]
tokens = [ablation_results[n]['avg_thinking_tokens'] for n in names]

# Accuracy comparison
colors = ['#2196F3' if n == 'Wait' else '#9E9E9E' for n in names]
bars = ax1.bar(names, accs, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Accuracy (%)', fontsize=11)
ax1.set_title(f'Trigger Phrase Ablation\n(n_wait={N_WAIT}, MATH500 {N_SAMPLES} samples)', fontsize=11)
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

# Avg thinking tokens
ax2.bar(names, tokens, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Avg Thinking Tokens', fontsize=11)
ax2.set_title('Average Thinking Tokens per Trigger', fontsize=11)

for ax in [ax1, ax2]:
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/trigger_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
with open('../results/trigger_ablation.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('Saved!')